# Baseline v3 — Legal HF + Cross-Encoder Reranking

Pipeline: **Legal_HF Retriever → Cross-Encoder Reranker**

| | Baseline v1 | Baseline v2 | **Baseline v3** | **v3 + Rerank** |
|--|--|--|--|--|
| Model | multilingual-MiniLM | PhoBERT | **Vietnam_legal** | **Vietnam_legal + CE** |
| Recall@1 | 0.2074 | 0.2167 | 0.3096 | ? |
| Recall@3 | 0.3529 | 0.3003 | 0.4458 | ? |
| Recall@5 | 0.4087 | 0.3591 | 0.4861 | ? |
| MRR@10 | 0.2908 | 0.2793 | 0.3924 | ? |

## Cell 0 — Imports & Config

In [1]:
import json, csv, time
import numpy as np
import faiss
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

# ── Paths ──
ROOT         = Path(".")
DATA_DIR     = ROOT / "data"
EVAL_DIR     = ROOT / "outputs" / "eval"
TMP_DIR      = ROOT / "outputs" / "tmp"
MDL_DIR      = ROOT / "outputs" / "models"

TRAIN_FILE    = DATA_DIR / "train.jsonl"
DEV_FILE      = DATA_DIR / "dev.jsonl"
TRAIN_NEG     = DATA_DIR / "train_with_neg.jsonl"
EVAL_QA_FILE  = EVAL_DIR / "eval_qa.jsonl"

FAISS_INDEX_V3 = TMP_DIR  / "faiss_v3.index"
FAISS_MAP_V3   = TMP_DIR  / "faiss_mapping_v3.jsonl"
RERANK_CSV_V1  = EVAL_DIR / "rerank_metrics.csv"       # v1 baseline
RERANK_CSV_V2  = EVAL_DIR / "rerank_metrics_v2.csv"    # v2 PhoBERT
RERANK_CSV_V3  = EVAL_DIR / "rerank_metrics_v3.csv"    # v3 Legal_HF (updated)

# ── Model config ──
BI_MODEL_NAME = "Quockhanh05/Vietnam_legal_embeddings"   # Bi-Encoder
CE_MODEL_PATH = MDL_DIR / "cross_encoder_v1" / "saved_model"  # Cross-Encoder
if not CE_MODEL_PATH.exists():
    CE_MODEL_PATH = MDL_DIR / "cross_encoder_v1"

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64 if DEVICE == "cuda" else 16
CE_BATCH   = 32
TOP_N      = 50

print(f"torch     : {torch.__version__}")
print(f"Device    : {DEVICE}")
print(f"Bi-Encoder: {BI_MODEL_NAME}")
print(f"CE path   : {CE_MODEL_PATH}")

d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch     : 2.6.0+cu124
Device    : cuda
Bi-Encoder: Quockhanh05/Vietnam_legal_embeddings
CE path   : outputs\models\cross_encoder_v1\saved_model


## Cell 1 — Utilities

In [2]:
def load_jsonl(path, max_rows=None):
    rows, errors = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows: break
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError: errors += 1
    if errors: print(f"  ⚠ {errors} malformed lines in {Path(path).name}")
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

def is_hit(faiss_id, expected_citations, mapping):
    row = mapping[faiss_id]
    for ec in expected_citations:
        ci = ec.get("chunk_index", -2)
        if ci != -1 and row["chunk_index"] == ci: return True
        if (row["van_ban"] == ec.get("van_ban","") and
            row["dieu"]    == ec.get("dieu",   "") and
            row["khoan"]   == ec.get("khoan",  "")): return True
    return False

def avg(lst): return round(sum(lst)/len(lst), 4) if lst else 0.0

print("Utilities loaded ✓")

Utilities loaded ✓


## Cell 2 — Load Models

In [3]:
# Load Bi-Encoder (Legal_HF)
print(f"Loading bi-encoder: {BI_MODEL_NAME}")
bi_model = SentenceTransformer(BI_MODEL_NAME, device=DEVICE)
print(f"  Bi-Encoder ready | dim={bi_model.get_sentence_embedding_dimension()}")

# Load Cross-Encoder
print(f"Loading cross-encoder: {CE_MODEL_PATH}")
ce_model = CrossEncoder(str(CE_MODEL_PATH), max_length=256, device=DEVICE)
print("  Cross-Encoder ready ✓")

Loading bi-encoder: Quockhanh05/Vietnam_legal_embeddings


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 813.81it/s, Materializing param=pooler.dense.weight]                               


  Bi-Encoder ready | dim=768
Loading cross-encoder: outputs\models\cross_encoder_v1\saved_model


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 724.76it/s, Materializing param=classifier.weight]                                    


  Cross-Encoder ready ✓


## Cell 3 — Build Corpus + FAISS Index

In [4]:
# Thu thập corpus
seen_passages = {}
for f in [TRAIN_FILE, DEV_FILE, TRAIN_NEG]:
    for r in load_jsonl(f):
        p = r.get("passage", "")
        if p and p not in seen_passages:
            meta = r.get("meta", {})
            seen_passages[p] = {
                "passage":     p,
                "chunk_index": meta.get("chunk_index", -1),
                "van_ban":     meta.get("van_ban",  ""),
                "chuong":      meta.get("chuong",   ""),
                "dieu":        meta.get("dieu",     ""),
                "khoan":       meta.get("khoan",    ""),
                "diem":        meta.get("diem",     ""),
            }

corpus  = list(seen_passages.values())
texts   = [c["passage"] for c in corpus]
print(f"Corpus: {len(corpus)} unique passages")

# Encode
print("Encoding with Legal_HF...")
t0 = time.perf_counter()
embeddings = bi_model.encode(
    texts, batch_size=BATCH_SIZE, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True
).astype("float32")
print(f"Shape: {embeddings.shape} | Time: {time.perf_counter()-t0:.1f}s")

# FAISS
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, str(FAISS_INDEX_V3))
print(f"FAISS → {FAISS_INDEX_V3}")

mapping = [{"faiss_id": i, **c} for i, c in enumerate(corpus)]
write_jsonl(FAISS_MAP_V3, mapping)
print(f"Mapping → {FAISS_MAP_V3} ({len(mapping)} entries)")

Corpus: 1861 unique passages
Encoding with Legal_HF...


Batches: 100%|██████████| 30/30 [00:14<00:00,  2.06it/s]

Shape: (1861, 768) | Time: 14.6s
FAISS → outputs\tmp\faiss_v3.index
Mapping → outputs\tmp\faiss_mapping_v3.jsonl (1861 entries)


## Cell 4 — Evaluate Baseline v3 + Reranked

In [5]:
eval_qa = load_jsonl(EVAL_QA_FILE)
print(f"Eval QA: {len(eval_qa)} questions")

r_base   = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}
r_rerank = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}

for item in tqdm(eval_qa, desc="Evaluate v3"):
    query = item["query"]
    ec    = item["expected_citations"]

    # Retrieve with Legal_HF
    q_emb = bi_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    _, ids = index.search(q_emb, TOP_N)
    ids    = ids[0].tolist()

    # ── Baseline metrics ──
    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        hit = any(is_hit(i, ec, mapping) for i in ids[:k] if i >= 0)
        r_base[key].append(1 if hit else 0)
    mrr = 0.0
    for rank, i in enumerate(ids[:10], 1):
        if i >= 0 and is_hit(i, ec, mapping): mrr = 1.0/rank; break
    r_base["MRR@10"].append(mrr)

    # ── Rerank with Cross-Encoder ──
    cands   = [(mapping[i]["passage"], i) for i in ids if i >= 0]
    rscores = ce_model.predict([[query, c[0]] for c in cands], batch_size=CE_BATCH) if cands else []
    ranked  = sorted(zip(rscores, [c[1] for c in cands]), reverse=True)
    r_ids   = [r[1] for r in ranked]

    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        hit = any(is_hit(i, ec, mapping) for i in r_ids[:k])
        r_rerank[key].append(1 if hit else 0)
    mrr = 0.0
    for rank, i in enumerate(r_ids[:10], 1):
        if is_hit(i, ec, mapping): mrr = 1.0/rank; break
    r_rerank["MRR@10"].append(mrr)

print("\n── Results (Legal_HF Baseline vs Reranked) ──")
print(f"  {'Metric':<10} {'Baseline v3':>14} {'v3+Reranked':>14} {'Δ':>8}")
print("  " + "-"*50)
for k, key in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]:
    bv3    = avg(r_base[k])
    reranked = avg(r_rerank[k])
    delta  = reranked - bv3
    sign   = "+" if delta >= 0 else ""
    print(f"  {key:<10} {bv3:>14.4f} {reranked:>14.4f} {sign}{delta:>7.4f}")

Eval QA: 323 questions


Evaluate v3: 100%|██████████| 323/323 [00:41<00:00,  7.76it/s]


── Results (Legal_HF Baseline vs Reranked) ──
  Metric        Baseline v3    v3+Reranked        Δ
  --------------------------------------------------
  Recall@1           0.3096         0.4768 + 0.1672
  Recall@3           0.4458         0.6099 + 0.1641
  Recall@5           0.4861         0.6409 + 0.1548
  MRR@10             0.3924         0.5501 + 0.1577


## Cell 5 — So sánh đầy đủ v1 / v2 / v3 / v3+Rerank & Lưu CSV

In [6]:
# Đọc v1 và v2
v1, v2 = {}, {}
if RERANK_CSV_V1.exists():
    with open(RERANK_CSV_V1, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            v1[row["metric"]] = float(row["baseline"])
if RERANK_CSV_V2.exists():
    with open(RERANK_CSV_V2, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            v2[row["metric"]] = float(row["baseline_v2_phobert"])

v3_base   = {"Recall@1":avg(r_base["R@1"]),   "Recall@3":avg(r_base["R@3"]),
             "Recall@5":avg(r_base["R@5"]),   "MRR@10":avg(r_base["MRR@10"])}
v3_rerank = {"Recall@1":avg(r_rerank["R@1"]), "Recall@3":avg(r_rerank["R@3"]),
             "Recall@5":avg(r_rerank["R@5"]), "MRR@10":avg(r_rerank["MRR@10"])}

print("\n" + "="*96)
print(f"  {'Metric':<10} {'v1(multilingual)':>18} {'v2(PhoBERT)':>13} {'v3(Legal_HF)':>14} {'v3+Rerank':>12} {'Δ(v3R-v1)':>11}")
print("="*96)
for metric in ["Recall@1","Recall@3","Recall@5","MRR@10"]:
    bv1    = v1.get(metric, float("nan"))
    bv2    = v2.get(metric, float("nan"))
    bv3    = v3_base[metric]
    bv3r   = v3_rerank[metric]
    delta  = bv3r - bv1
    sign   = "+" if delta >= 0 else ""
    print(f"  {metric:<10} {bv1:>18.4f} {bv2:>13.4f} {bv3:>14.4f} {bv3r:>12.4f} {sign}{delta:>10.4f}")
print("="*96)

# Lưu CSV
rows_v3 = [
    {"metric":"Recall@1","v1_multilingual":v1.get("Recall@1",""),"v2_phobert":v2.get("Recall@1",""),"v3_legal_hf":v3_base["Recall@1"],"v3_reranked":v3_rerank["Recall@1"]},
    {"metric":"Recall@3","v1_multilingual":v1.get("Recall@3",""),"v2_phobert":v2.get("Recall@3",""),"v3_legal_hf":v3_base["Recall@3"],"v3_reranked":v3_rerank["Recall@3"]},
    {"metric":"Recall@5","v1_multilingual":v1.get("Recall@5",""),"v2_phobert":v2.get("Recall@5",""),"v3_legal_hf":v3_base["Recall@5"],"v3_reranked":v3_rerank["Recall@5"]},
    {"metric":"MRR@10",  "v1_multilingual":v1.get("MRR@10",  ""),"v2_phobert":v2.get("MRR@10",  ""),"v3_legal_hf":v3_base["MRR@10"],  "v3_reranked":v3_rerank["MRR@10"]},
]
with open(RERANK_CSV_V3, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["metric","v1_multilingual","v2_phobert","v3_legal_hf","v3_reranked"])
    w.writeheader(); w.writerows(rows_v3)

print(f"\nSaved → {RERANK_CSV_V3} ✓")


  Metric       v1(multilingual)   v2(PhoBERT)   v3(Legal_HF)    v3+Rerank   Δ(v3R-v1)
  Recall@1               0.2074        0.2167         0.3096       0.4768 +    0.2694
  Recall@3               0.3529        0.3003         0.4458       0.6099 +    0.2570
  Recall@5               0.4087        0.3591         0.4861       0.6409 +    0.2322
  MRR@10                 0.2908        0.2793         0.3924       0.5501 +    0.2593

Saved → outputs\eval\rerank_metrics_v3.csv ✓
